# YOLO26-P2 + Color Consistency Loss — BSTLD Tiny Traffic Light Detection

## Novel Contributions
1. **YOLO26-P2**: thêm P2 head (stride 4) vào YOLO26 → phát hiện đèn chỉ 8px
2. **Color Consistency Loss**: auxiliary loss vật lý màu sắc — ép model học đúng màu HSV của từng class

## Architecture & Training Stack
| Component | Chi tiết |
|---|---|
| Backbone | YOLO26s (pretrained COCO) |
| Detection heads | P2/4 + P3/8 + P4/16 + P5/32 |
| Small obj training | STAL + ProgLoss (YOLO26 built-in) |
| Auxiliary loss | Color Consistency (HSV physics) λ=0.08 |
| Optimizer | MuSGD (YOLO26 built-in) |
| Input resolution | 1280×1280 |

## Ước tính thời gian (YOLO26s-P2, imgsz=1280)
| GPU | Thời gian/epoch | 100 epochs |
|-----|----------------|-----------|
| T4 (free) | ~5-7 phút | ~9-12 giờ (cần resume) |
| A100 (Pro) | ~2-3 phút | ~3.5-5 giờ |

Early stop (patience=40) thường dừng lúc ~80-100 epoch.

## BƯỚC 0 — Chuẩn bị & Upload file lên Drive

### Re-nén bstld_config.zip với các file mới (chạy PowerShell từ apps/worker/)

```powershell
# Xóa config zip cũ
Remove-Item bstld_config.zip -ErrorAction SilentlyContinue

# Nén lại với yolo26_p2.yaml + bstld.yaml + color_consistency.py
$7z = "C:\Program Files\7-Zip\7z.exe"
& $7z a -tzip bstld_config.zip `
    models\yolo26_p2.yaml `
    datasets\bstld.yaml `
    scripts\color_consistency.py
```

### Files cần upload lên `My Drive/SDO_train/`

| File | Nội dung | Size |
|------|----------|------|
| `bstld_config.zip` | yolo26_p2.yaml + bstld.yaml + color_consistency.py | ~5 KB |
| `bstld_train.zip.001`, `.002`, `.003` | Train images + labels | ~6.2 GB |
| `bstld_val.zip.001`, `.002` | Val 2000 images + labels | ~2.9 GB |

Upload ~10GB: khoảng **60-120 phút** tuỳ kết nối.

In [ ]:
# ── CELL 1: Kiểm tra GPU ──────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], 
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip())

import torch
print('CUDA available:', torch.cuda.is_available())
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

In [ ]:
# ── CELL 2: Cài thư viện ─────────────────────────────────────────────────
!pip install -q ultralytics
import ultralytics
print('Ultralytics version:', ultralytics.__version__)

In [ ]:
# ── CELL 3: Mount Google Drive ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
# Điều chỉnh đường dẫn Drive nếu cần
DRIVE_DIR = '/content/drive/MyDrive/SDO_train'
print('Files in Drive folder:')
print(os.listdir(DRIVE_DIR))

In [ ]:
# ── CELL 4: Giải nén vào ổ local Colab (/content) ──────────────────────
# QUAN TRỌNG: KHÔNG đọc ảnh trực tiếp từ Drive — giải nén vào /content/ trước.
# File được chia nhỏ bằng 7z: .zip.001, .zip.002... → dùng subprocess 7z để giải nén.

import os, subprocess, time, zipfile

WORK_DIR = '/content/bstld'
os.makedirs(WORK_DIR, exist_ok=True)

def extract_7z_split(drive_dir, base_name, dest_dir):
    """Giải nén file 7z split (.zip.001, .zip.002...) hoặc file .zip thường."""
    # Kiểm tra xem là split hay single
    part1 = os.path.join(drive_dir, base_name + '.001')
    single = os.path.join(drive_dir, base_name)

    if os.path.exists(part1):
        first_part = part1
        print(f'  Detected split archive: {base_name}.001, .002...')
    elif os.path.exists(single):
        first_part = single
        print(f'  Detected single archive: {base_name}')
    else:
        print(f'  WARNING: {base_name} not found in {drive_dir}')
        return

    t0 = time.time()
    # Cài 7z nếu chưa có
    if not os.path.exists('/usr/bin/7z'):
        print('  Installing p7zip...')
        subprocess.run(['apt-get', 'install', '-y', '-q', 'p7zip-full'], check=True)

    r = subprocess.run(['7z', 'x', first_part, f'-o{dest_dir}', '-y'],
                       capture_output=True, text=True)
    elapsed = time.time() - t0
    if r.returncode == 0:
        print(f'  Done in {elapsed:.0f}s')
    else:
        print(f'  ERROR (code {r.returncode}):')
        print(r.stderr[-500:])

# 1. Config (single zip)
print('Extracting config...')
with zipfile.ZipFile(os.path.join(DRIVE_DIR, 'bstld_config.zip')) as z:
    z.extractall(WORK_DIR)
print('  Done')

# 2. Train images (split zip)
print('Extracting train...')
extract_7z_split(DRIVE_DIR, 'bstld_train.zip', WORK_DIR)

# 3. Val images (split zip)
print('Extracting val...')
extract_7z_split(DRIVE_DIR, 'bstld_val.zip', WORK_DIR)

# Kiem tra
import glob, sys
pngs_train = len(glob.glob(f'{WORK_DIR}/dataset_train_rgb/**/*.png', recursive=True))
pngs_val   = len(glob.glob(f'{WORK_DIR}/dataset_val_sample/**/*.png', recursive=True))
print(f'\nTrain PNGs: {pngs_train} | Val PNGs: {pngs_val}')
if pngs_train < 5000:
    print('WARNING: Train PNGs count low — check if all .zip.00x parts were uploaded')
if pngs_val < 1500:
    print('WARNING: Val PNGs count low — check if all .zip.00x parts were uploaded')

# Verify custom scripts are present
assert os.path.exists(f'{WORK_DIR}/yolo26_p2.yaml'), 'yolo26_p2.yaml missing from config zip!'
assert os.path.exists(f'{WORK_DIR}/color_consistency.py'), 'color_consistency.py missing from config zip!'
print('\nAll required files present.')

# Make scripts importable
sys.path.insert(0, WORK_DIR)

In [ ]:
# ── CELL 5: Tạo bstld.yaml — tự động tìm đúng path ────────────────────
import yaml, glob as _glob, os

def find_png_root(base, hint_parts):
    """Tìm thư mục chứa PNG gần nhất khớp với hint_parts."""
    for part in hint_parts:
        candidate = os.path.join(base, part)
        if os.path.isdir(candidate):
            pngs = _glob.glob(candidate + '/**/*.png', recursive=True)
            if pngs:
                return candidate
    # Fallback: tìm bất kỳ thư mục nào có PNG
    for root, dirs, files in os.walk(base):
        if any(f.endswith('.png') for f in files):
            return root
    return None

# Tìm train dir
TRAIN_HINTS = [
    'dataset_train_rgb/rgb/train',
    'rgb/train',
    'dataset_train_rgb',
]
VAL_HINTS = [
    'dataset_val_sample/rgb/val',
    'dataset_val_sample/rgb',
    'dataset_val_sample',
    'rgb/val',
]

train_root = find_png_root(WORK_DIR, TRAIN_HINTS)
val_root   = find_png_root(WORK_DIR, VAL_HINTS)

assert train_root, f'Could not find train PNGs under {WORK_DIR}'
assert val_root,   f'Could not find val PNGs under {WORK_DIR}'

# Chuyển về path tương đối so với WORK_DIR
train_rel = os.path.relpath(train_root, WORK_DIR)
val_rel   = os.path.relpath(val_root,   WORK_DIR)

print(f'Train dir: {train_root}  ({len(_glob.glob(train_root+"/**/*.png",recursive=True))} PNGs)')
print(f'Val   dir: {val_root}    ({len(_glob.glob(val_root  +"/**/*.png",recursive=True))} PNGs)')

dataset_cfg = {
    'path':  WORK_DIR,
    'train': train_rel,
    'val':   val_rel,
    'nc': 4,
    'names': {0: 'red', 1: 'yellow', 2: 'green', 3: 'off'},
}

DATASET_YAML = f'{WORK_DIR}/bstld.yaml'
with open(DATASET_YAML, 'w') as f:
    yaml.dump(dataset_cfg, f, default_flow_style=False)

print('\nDataset config:')
print(open(DATASET_YAML).read())

In [ ]:
# ── CELL 6: TRAIN YOLO26-P2 + Color Consistency Loss ─────────────────────
#
# FULL MODEL STACK:
#   ① YOLO26s-P2   — 4-scale detection (stride 4/8/16/32), backbone pretrained
#   ② STAL          — built-in: tiny objects (<8px) get min 4 anchor assignments
#   ③ ProgLoss      — built-in: progressive loss balancing
#   ④ Color Loss    — novel: HSV physics auxiliary loss (λ=0.08, starts epoch 15)
#   ⑤ MuSGD        — built-in: fast stable convergence
#
# Ước tính thời gian (YOLO26s-P2, imgsz=1280):
#   T4  : ~5-7 min/epoch → 100 epochs ≈ 9-12 giờ (cần resume)
#   A100: ~2-3 min/epoch → 100 epochs ≈ 3.5-5 giờ

import importlib, color_consistency
importlib.reload(color_consistency)          # đảm bảo dùng bản mới nhất
from color_consistency import make_trainer

RUN_DIR  = f'{DRIVE_DIR}/runs'
RUN_NAME = 'yolo26p2_color_bstld_1280'

trainer = make_trainer(
    model_yaml   = f'{WORK_DIR}/yolo26_p2.yaml',
    data_yaml    = DATASET_YAML,
    pretrained   = 'yolo26s.pt',
    imgsz        = 1280,
    epochs       = 150,
    patience     = 40,
    batch        = -1,
    project      = RUN_DIR,
    name         = RUN_NAME,
    lambda_color = 0.08,
    color_warmup = 15,
    min_box_area = 16,
    # extra YOLO args
    device       = 0,
)

trainer.train()
BEST_PT = f'{RUN_DIR}/{RUN_NAME}/weights/best.pt'
print('Training complete! Best weights:', BEST_PT)

In [ ]:
# ── CELL 7: Resume khi session bị ngắt ──────────────────────────────────
# Chỉ chạy cell này khi cần resume. KHÔNG chạy khi train vừa xong.

# from color_consistency import ColorConsistencyTrainer
# LAST_PT = f'{RUN_DIR}/{RUN_NAME}/weights/last.pt'
# trainer = ColorConsistencyTrainer(overrides={
#     'model':   LAST_PT,
#     'resume':  True,
#     'device':  0,
# })
# trainer.train()

In [ ]:
# ── CELL 8: Đánh giá mAP trên tập val ────────────────────────────────────
from ultralytics import YOLO
import json

BEST_PT = f'{RUN_DIR}/{RUN_NAME}/weights/best.pt'
model = YOLO(BEST_PT)

metrics = model.val(
    data=DATASET_YAML,
    imgsz=1280,
    batch=8,
    conf=0.001,     # Ngưỡng thấp để đánh giá recall đầy đủ
    iou=0.5,
    device=0,
)

print('\n=== Metrics Summary ===')
print(f'mAP@50        : {metrics.box.map50:.4f}')
print(f'mAP@50-95     : {metrics.box.map:.4f}')
print(f'Precision     : {metrics.box.mp:.4f}')
print(f'Recall        : {metrics.box.mr:.4f}')

print('\n=== Per-class AP@50 ===')
class_names = ['red', 'yellow', 'green', 'off']
for i, (ap, name) in enumerate(zip(metrics.box.ap50, class_names)):
    print(f'  {name:8s}: AP@50 = {ap:.4f}')

print('\n=== Inference Speed (val set) ===')
# metrics.speed: {'preprocess': ms, 'inference': ms, 'postprocess': ms}
for k, v in metrics.speed.items():
    print(f'  {k:15s}: {v:.2f} ms/img')
total_ms = sum(metrics.speed.values())
print(f'  {"Total":15s}: {total_ms:.2f} ms/img  →  {1000/total_ms:.1f} FPS')

In [ ]:
# ── CELL 9 (NEW): Ablation Study — quan trọng cho paper ──────────────────
# So sánh 3 cấu hình để chứng minh từng contribution có tác dụng:
#
#   A: YOLO26s baseline           (không P2, không Color Loss)
#   B: YOLO26s-P2                 (có P2, không Color Loss)
#   C: YOLO26s-P2 + Color Loss    (full method — đã train ở Cell 6)
#
# Chỉ chạy sau khi đã có best.pt từ Cell 6.
# QUAN TRỌNG: A và B cần train riêng (lần lượt chạy cell này).

from ultralytics import YOLO
from color_consistency import ColorConsistencyTrainer
import json

ABLATION_RESULTS = {}

def run_ablation(name, model_src, use_color_loss=False):
    print(f'\n{"="*60}\nAblation: {name}\n{"="*60}')
    overrides = {
        'data':         DATASET_YAML,
        'imgsz':        1280,
        'epochs':       80,      # shorter for ablation
        'patience':     30,
        'batch':        -1,
        'device':       0,
        'amp':          True,
        'cache':        'disk',
        'workers':      4,
        'mosaic':       1.0,
        'close_mosaic': 10,
        'scale':        0.2,
        'hsv_h':        0.01,
        'hsv_s':        0.7,
        'hsv_v':        0.4,
        'flipud':       0.0,
        'fliplr':       0.5,
        'copy_paste':   0.1,
        'project':      f'{RUN_DIR}/ablation',
        'name':         name,
        'exist_ok':     True,
        'save':         True,
    }
    if use_color_loss:
        overrides.update({'model': model_src, 'pretrained': 'yolo26s.pt',
                          'lambda_color': 0.08, 'color_warmup': 15})
        t = ColorConsistencyTrainer(overrides=overrides)
        t.train()
        best = f'{RUN_DIR}/ablation/{name}/weights/best.pt'
    else:
        overrides['model'] = model_src
        if model_src.endswith('.yaml'):
            overrides['pretrained'] = 'yolo26s.pt'
        t = ColorConsistencyTrainer(overrides={**overrides, 'lambda_color': 0.0, 'color_warmup': 9999})
        t.train()
        best = f'{RUN_DIR}/ablation/{name}/weights/best.pt'

    # Evaluate
    model = YOLO(best)
    m = model.val(data=DATASET_YAML, imgsz=1280, conf=0.001, iou=0.5, device=0)
    ABLATION_RESULTS[name] = {
        'mAP50':     round(m.box.map50, 4),
        'mAP50-95':  round(m.box.map,   4),
        'precision': round(m.box.mp,    4),
        'recall':    round(m.box.mr,    4),
        'AP50_per_class': {
            cls: round(ap, 4)
            for cls, ap in zip(['red','yellow','green','off'], m.box.ap50)
        }
    }
    print(f'{name}: mAP50={ABLATION_RESULTS[name]["mAP50"]}')

# Uncomment ONE at a time to run ablation:

# ── A: Baseline ──────────────────────────────────────────────────────────
# run_ablation('A_yolo26s_baseline', 'yolo26s.pt', use_color_loss=False)

# ── B: P2 only ───────────────────────────────────────────────────────────
# run_ablation('B_yolo26s_p2', f'{WORK_DIR}/yolo26_p2.yaml', use_color_loss=False)

# ── C: Full (already trained in Cell 6 — just evaluate) ──────────────────
# from ultralytics import YOLO
# model = YOLO(BEST_PT)
# m = model.val(data=DATASET_YAML, imgsz=1280, conf=0.001, iou=0.5, device=0)
# ABLATION_RESULTS['C_yolo26s_p2_color'] = {...}

# ── Print final ablation table ────────────────────────────────────────────
print('\n=== ABLATION TABLE ===')
print(f'{"Method":<30} {"mAP50":>7} {"mAP50-95":>9} {"P":>7} {"R":>7}')
print('-' * 62)
for name, r in ABLATION_RESULTS.items():
    print(f'{name:<30} {r["mAP50"]:>7.4f} {r["mAP50-95"]:>9.4f} '
          f'{r["precision"]:>7.4f} {r["recall"]:>7.4f}')

# Save to Drive
with open(f'{RUN_DIR}/ablation_results.json', 'w') as f:
    json.dump(ABLATION_RESULTS, f, indent=2)
print('\nSaved ablation results to Drive.')

In [ ]:
# ── CELL 9: Export về máy ─────────────────────────────────────────────────
# best.pt đã được lưu vào Drive ở CELL 6 (project=DRIVE_DIR/runs/...).
# Copy thêm 1 bản vào root Drive cho tiện download:
import shutil
shutil.copy(BEST_PT, f'{DRIVE_DIR}/best_yolo26p2_color_bstld.pt')
print('Saved to Drive:', f'{DRIVE_DIR}/best_yolo26p2_color_bstld.pt')

# Sau đó trên máy Windows:
# 1. Download file best_bstld.pt từ Google Drive
# 2. Đặt vào: apps/worker/yolov8_p2_custom_best.pt  (ghi đè file cũ)
# 3. Restart worker service là xong

## Sau khi train xong — tích hợp vào pipeline SDO

1. Download `best_bstld.pt` từ Drive về máy
2. Đặt vào `apps/worker/yolov8_p2_custom_best.pt` (ghi đè)
3. Cập nhật `apps/worker/app/config.py`: đổi mapping class từ 3 → 4 (thêm `off`)
4. Restart worker: `uvicorn app.main:app --reload --port 8000`

**Lưu ý quan trọng về pipeline:**
Model mới train trực tiếp trên ảnh full-frame (không qua crop+upscale), 
nên sẽ cần điều chỉnh `recursion_engine.py` để inference single-pass full-frame 
thay vì đưa crop vào P2 như hiện tại. Đây là Phase 5 sau khi có weights mới.